# Classification pipeline walkthrough

End-to-end visualisation of every preprocessing step that takes a raw
ISFET binary recording and produces the per-pixel input that the seven
Paper 3 models train on.

This notebook walks through ONE concrete example (chip `1e9`,
well `0`) so every stage can be inspected. Re-run with a
different `CHIP_KEY` or `WELL_IDX` at the top to inspect a different
chip/well.

**Outline**
1. Setup + pick one chip
2. Raw binary import (titan)
3. Linearised voltage cube
4. Active-pixel masks (firmware → QC → titan-strict)
5. Per-pixel time series for one well
6. Time axis + idx_settled / idx_start
7. Crop to 450 samples post-amplification-onset
8. Trapped-charge subtraction
9. The cached dataset — class balance, per-chip distribution
10. Train / val / test splits — random vs chip-fold
11. Model input — raw 1D vs STFT spectrogram
12. What the 2D-CNN actually sees


## 1. Setup

Import the project's classification module so we share constants with the real pipeline. titan is on `sys.path` automatically.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Walk up to the project root so we can import the module by name
HERE = Path.cwd()
for _ in range(5):
    if (HERE / "Analysis" / "classification").exists():
        break
    HERE = HERE.parent
sys.path.insert(0, str(HERE))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram

from Analysis.classification import paths
from Analysis.classification.data import build_dataset
from Analysis.classification.features import spectrogram as spec_module

CHIP_KEY = "1e9"
WELL_IDX = 0
chip_path = paths.FINAL_CHIPS[CHIP_KEY]
print("Chip path:", chip_path.name)
print("Cache path:", paths.cache_path("dataset_per_pixel"))


## 2. Raw binary import via titan

titan reads the chip's `.bin` files and returns an `Experiment` with one `Well` per partition of the 290×204 sensor array. We pull `well 0` so we can show every downstream stage on the same well.

In [ ]:
from titan.load_and_preprocessing import titan_load_and_preprocessing

# start_type='temperature' tells titan to find idx_settled by spotting the
# temperature plateau, which skips the reference-electrode pulse window at
# the recording start. Same setting used by the supervisor's quantification
# code; required so the pre-amplification reference pulse doesn't leak
# into the model input (per-pixel pulse signatures are chip-identifying).
exp = titan_load_and_preprocessing(
    chip_path,
    n_wells=paths.N_WELLS,
    start_type="temperature",
    n_a_type="v01",
    end_time_min=40,
    print_status=False,
)
well = exp.wells_list[WELL_IDX]
print(f"Loaded {chip_path.name} -> {len(exp.wells_list)} wells")
print(f"well {WELL_IDX}:")
print(f"  well_3d shape (rows, cols, time)        : {well.well_3d.shape}")
print(f"  time_npr.shape (full time vector)       : {well.time_npr.shape}")
print(f"  idx_settled  = {well.idx_settled}  (= {well.time_npr[well.idx_settled]/60:.2f} min absolute)")
print(f"  idx_start    = {well.idx_start}    (= {well.time_npr[well.idx_start]/60:.2f} min absolute)")
print(f"  idx_end      = {well.idx_end}      (= {well.time_npr[well.idx_end-1]/60:.2f} min absolute)")
dt_sec = float(well.time_npr[1] - well.time_npr[0])
print(f"  sampling dt  = {dt_sec:.2f} sec/sample")


## 3. Linearised voltage cube

`well.well_3d` is the raw voltage cube (rows × cols × time) AFTER titan's
linearisation. **Most pixels are dead** — never wired up by the firmware
or linearisation failed → output stuck at zero or some constant. About
68% of pixels on this chip are dead this way; only ~32% (the
firmware-active ones) carry real signal.

To demonstrate this, the right column below plots 6 *random* pixels
(no filtering). They look completely flat because most rolls of the dice
land on dead pixels. The bottom-right plot picks 6 from the **firmware
active mask only** — those carry the real ISFET response. This is
exactly why the next section's active-pixel filter is necessary.

In [ ]:
v_cube = well.well_3d            # (rows, cols, time), linearised
n_rows, n_cols, n_time = v_cube.shape
mid_t = n_time // 2

# Need the firmware mask to pick "real" pixels for the second panel
from Analysis.classification.data import build_dataset
fw_per_well = build_dataset._loose_active_mask_per_well(chip_path)
fw_flat = fw_per_well[WELL_IDX]                # 1D mask over n_rows*n_cols
fw_indices = np.where(fw_flat)[0]
print(f"Total pixels in well: {fw_flat.size}   "
      f"firmware-active: {fw_flat.sum()}   "
      f"dead: {(~fw_flat).sum()}")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
t_min = (well.time_npr[:n_time] - well.time_npr[0]) / 60.0

# Top-left: voltage heatmap mid-experiment
ax = axes[0, 0]
im = ax.imshow(v_cube[:, :, mid_t], cmap="viridis", aspect="auto")
ax.set(title=f"Voltage at sample {mid_t} (V)", xlabel="col", ylabel="row")
plt.colorbar(im, ax=ax, label="V")

# Top-right: where the firmware mask says pixels are active
ax = axes[0, 1]
ax.imshow(fw_flat.reshape(n_rows, n_cols, order="C"),
          cmap="gray_r", aspect="auto")
ax.set(title=f"Firmware-active mask\n({int(fw_flat.sum())} of "
             f"{fw_flat.size} pixels)", xlabel="col", ylabel="row")

# Bottom-left: 6 random pixels (no filtering) - mostly DEAD, looks flat
rng = np.random.default_rng(0)
ax = axes[1, 0]
flat_idx = rng.choice(n_rows * n_cols, size=6, replace=False)
n_dead = 0
for i in flat_idx:
    r, c = i // n_cols, i % n_cols
    is_dead = "DEAD" if not fw_flat[i] else "ACTIVE"
    if is_dead == "DEAD":
        n_dead += 1
    ax.plot(t_min, v_cube[r, c, :], lw=0.8,
            label=f"({r},{c}) {is_dead}")
ax.set(xlabel="Time [min]", ylabel="V (linearised)",
       title=f"6 random pixels - no filter ({n_dead}/6 dead = flat)")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

# Bottom-right: 6 firmware-active pixels - real ISFET responses
rng2 = np.random.default_rng(0)
ax = axes[1, 1]
active_pick = rng2.choice(fw_indices, size=6, replace=False)
for i in active_pick:
    r, c = i // n_cols, i % n_cols
    ax.plot(t_min, v_cube[r, c, :], lw=0.8, label=f"({r},{c})")
ax.set(xlabel="Time [min]", ylabel="V (linearised)",
       title="6 firmware-active pixels - real ISFET signals")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

plt.tight_layout()


## 4. Active-pixel masks — what we use, and how it compares to titan

For per-pixel training we need a clean active-pixel set. Our pipeline
applies a Paper-3-style mask that is closer to the criteria Paper 3
described than to titan's stricter per-experiment filter:

**Our mask (used by build_dataset.py):**
- firmware-active flag from `find_active.bin` (the chip's own "this
  pixel is wired up" signal)
- × finite values throughout (no NaN/inf from failed linearisation)
- × first-sample voltage in `[0.5, 5.0]` V (drop dead/saturated)
- × signal std > 1e-3 (drop flat-zero pixels)

**titan's `idx_active` for reference:** chains firmware-active +
not-temp-pixel + gain-OK + linearisation-OK + per-experiment vrange +
derivative bounds. titan's extra checks were tuned for well-averaged
quantification (where retaining a few clean pixels per well is fine);
per-pixel ML wants a larger active set, which our filter delivers
(typically ~1.2-1.5× more pixels than titan's strict mask).

The four heatmaps below show each mask stage so you can see what gets
included/excluded at each step.

In [ ]:
# Firmware mask (loaded directly via the same helper used by build_dataset.py)
loose_per_well = build_dataset._loose_active_mask_per_well(chip_path)
fw_mask = loose_per_well[WELL_IDX]                      # (n_rows*n_cols,)
fw_mask_2d = fw_mask.reshape(n_rows, n_cols, order="C")

# QC mask
v_2d = well.well_2d                                     # (T, n_rows*n_cols)
qc_mask = build_dataset._qc_mask(v_2d)
qc_mask_2d = qc_mask.reshape(n_rows, n_cols, order="C")

# Combined (this is what build_dataset uses)
combined = fw_mask & qc_mask
combined_2d = combined.reshape(n_rows, n_cols, order="C")

# titan's strict mask, for reference
titan_strict = well.idx_active
titan_strict_2d = titan_strict.reshape(n_rows, n_cols, order="C")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, m, title, count in [
    (axes[0], fw_mask_2d,        "Firmware-active", fw_mask.sum()),
    (axes[1], qc_mask_2d,        "QC-pass alone",   qc_mask.sum()),
    (axes[2], combined_2d,       "Firmware ∩ QC (used)", combined.sum()),
    (axes[3], titan_strict_2d,   "titan strict idx_active", titan_strict.sum()),
]:
    ax.imshow(m, cmap="gray_r", aspect="auto")
    ax.set(title=f"{title}\n({count} pixels)")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f"Active-pixel masks for {chip_path.name} well {WELL_IDX}")
plt.tight_layout()
print(f"firmware     : {fw_mask.sum():>5}  / {fw_mask.size}")
print(f"QC alone     : {qc_mask.sum():>5}  / {qc_mask.size}")
print(f"firmware ∩QC : {combined.sum():>5}  / {combined.size}  <-- what we use (build_dataset.py)")
print(f"titan strict : {titan_strict.sum():>5}  / {titan_strict.size}  <-- titan's idx_active (for reference)")


## 5. Per-pixel time series for one well

Now we apply the combined mask to `well.well_2d` and show all surviving per-pixel traces overlaid (transparent), plus a few highlighted ones.

In [ ]:
v_active = v_2d[:, combined]    # (T, N_active)
n_active = v_active.shape[1]
print(f"After mask: {v_active.shape[1]} active pixel traces")

t_well = well.time_min          # 0 = idx_start; goes negative before
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_well, v_active, color="C0", alpha=0.02)
# Highlight a few
hl_idx = np.linspace(0, n_active - 1, 5).astype(int)
for j in hl_idx:
    ax.plot(t_well, v_active[:, j], lw=1.5,
            label=f"pixel #{j}")
ax.axvline(0, color="r", ls="--", alpha=0.6, label="idx_start (time = 0)")
ax.set(xlabel="Time [min]", ylabel="V (linearised)",
       title=f"Well {WELL_IDX} — {n_active} active per-pixel traces")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()


## 6. Time axis + key indices

titan's `time_npr` is in seconds since file start. `time_min` is the well's view: 0 = `idx_start` (= the moment titan calls amplification onset), negative for the temperature-settling phase before, positive after. Sampling rate is ~3.3 sec/sample for these chips so 450 samples covers ~25 minutes.

In [ ]:
print(f"time_min[0]                   = {well.time_min[0]:>7.2f}  (idx_settled, before onset)")
print(f"time_min where time_min >= 0  = onset (= idx_start)")
print(f"time_min[-1]                  = {well.time_min[-1]:>7.2f}  (idx_end)")

onset_idx = int(np.searchsorted(well.time_min, 0.0))
post = well.time_min[onset_idx:onset_idx + 450]
print()
print(f"onset_idx (first sample with time_min >= 0) = {onset_idx}")
print(f"450 samples post-onset cover [{post[0]:.2f}, {post[-1]:.2f}] min relative to onset")
print(f"= {(post[-1] - post[0]):.2f} minutes wall-clock")


## 7. Crop to 450 samples post-onset (Paper 3 §IV.A)

All training inputs use the first 450 samples *after* amplification onset. The pre-onset temperature-settling phase is dropped (it would leak temperature transients the model could memorise).

In [ ]:
v_post = v_active[onset_idx:onset_idx + 450, :]    # (450, N_active)

fig, ax = plt.subplots(figsize=(12, 4))
t_post = post - post[0]    # 0..~25 min relative to onset
ax.plot(t_post, v_post, color="C0", alpha=0.02)
for j in hl_idx:
    ax.plot(t_post, v_post[:, j], lw=1.5, label=f"pixel #{j}")
ax.set(xlabel="Time since onset [min]", ylabel="V (linearised)",
       title=f"Cropped to 450 samples post-onset ({v_post.shape})")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()


## 8. Trapped-charge subtraction

Each pixel has an unknown DC offset due to manufacturing variation
(trapped charge). Paper 3 strips this by subtracting the first
post-onset sample so every pixel starts at 0. This is the same simple
correction Paper 3 acknowledges as crude — it removes the offset but
not the per-pixel drift rate variation.

In [ ]:
v_corrected = v_post - v_post[0:1, :]     # (450, N_active), starts at 0

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(t_post, v_post, color="C0", alpha=0.05)
for j in hl_idx:
    ax.plot(t_post, v_post[:, j], lw=1.5)
ax.set(title="Before (raw, with trapped charge)",
       xlabel="Time since onset [min]", ylabel="V")
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(t_post, v_corrected, color="C2", alpha=0.05)
for j in hl_idx:
    ax.plot(t_post, v_corrected[:, j], lw=1.5)
ax.axhline(0, color="k", lw=0.5)
ax.set(title="After trapped-charge subtraction (= model input)",
       xlabel="Time since onset [min]", ylabel="ΔV")
ax.grid(alpha=0.3)

plt.suptitle(f"Well {WELL_IDX}: every pixel pinned to 0 at onset")
plt.tight_layout()
print(f"Final per-pixel input shape (one well): {v_corrected.shape}")
print(f"This stack of 450-sample traces is what the cache stores.")


## 9. The cached dataset

Above we walked through ONE chip + ONE well. The full pipeline runs over every chip + well in scope and concatenates into a single `(N_pixels, 450)` matrix `X` plus aligned `y`, `chip_id`, `well_id`, `pixel_id`. Below: load the cache and inspect class balance, per-chip distribution, and what positives vs NTCs actually look like.

In [ ]:
cache = np.load(paths.cache_path("dataset_per_pixel"), allow_pickle=False)
print("Cache keys:", list(cache.keys()))
X, y = cache["X"], cache["y"]
chip_id, well_id = cache["chip_id"], cache["well_id"]
print(f"X.shape = {X.shape}   y.shape = {y.shape}")
print(f"Positives = {(y == 1).sum():,}   NTCs = {(y == 0).sum():,}   "
      f"ratio = {(y == 1).sum() / max((y == 0).sum(), 1):.2f}:1")

# Per-chip pixel count
unique_chips = np.unique(chip_id)
print(f"\nChips in cache: {len(unique_chips)}")
for c in unique_chips[:8]:
    mask = chip_id == c
    n_pos = int(((y == 1) & mask).sum())
    n_neg = int(((y == 0) & mask).sum())
    print(f"  {c[:55]:55}  pos={n_pos:>5}  neg={n_neg:>5}")
if len(unique_chips) > 8:
    print(f"  ... ({len(unique_chips)-8} more chips)")


### Positive vs NTC traces — what the model sees as 'amplification'

Random sample of 50 positive and 50 NTC pixel traces from the cache. If the cache is correct, positives should sigmoidally rise; NTCs should be flatter / monotonically drifty.

In [ ]:
rng = np.random.default_rng(42)
pos_idx = rng.choice(np.where(y == 1)[0], size=50, replace=False)
neg_idx = rng.choice(np.where(y == 0)[0], size=50, replace=False)

t_axis = np.arange(450) * dt_sec / 60.0    # ~3.3 sec/sample → minutes
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
axes[0].plot(t_axis, X[pos_idx].T, color="C2", alpha=0.4)
axes[0].set(title="50 random POSITIVE pixel traces",
            xlabel="Time since onset [min]", ylabel="ΔV (post trap-charge sub)")
axes[0].grid(alpha=0.3)
axes[1].plot(t_axis, X[neg_idx].T, color="C3", alpha=0.4)
axes[1].set(title="50 random NTC pixel traces",
            xlabel="Time since onset [min]")
axes[1].grid(alpha=0.3)
plt.tight_layout()


## 10. Train / val / test splits

The cache supports two splits:

- **Random pixel-level 70/15/15 (Paper 3 fidelity).** Stratified by
  class. Pixels from the same well end up scattered across train/val/test
  → leaky for our 5-chip dataset, gives optimistic numbers (`exp1_final_random`).
- **Leave-one-chip-out chip-fold CV.** Each fold holds out a whole
  chip's pixels for test. Honest generalisation (`exp2_final_chipfold`).

The two visualisations below show how each split partitions the chips.

In [ ]:
from Analysis.classification.data import dataset

# --- Split A: random 70/15/15 ---
arr = dataset.make_split(features="raw", split="random", seed=0)
print(f"Random split  : train {len(arr.y_tr)}  val {len(arr.y_va)}  "
      f"test {len(arr.y_te)}")
print(f"  Test chips represented: "
      f"{len(np.unique(arr.chip_te))} (out of {len(np.unique(chip_id))} in cache)")

# Counts per chip in test set
tcount = {c: int((arr.chip_te == c).sum()) for c in np.unique(arr.chip_te)}
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(len(tcount)), list(tcount.values()), color="C0")
ax.set_xticks(range(len(tcount)))
ax.set_xticklabels([c[:25] for c in tcount], rotation=70, ha="right",
                   fontsize=8)
ax.set(title="Random split — test pixels per chip (every chip leaks into test)",
       ylabel="# test pixels")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()


In [ ]:
# --- Split B: chip-fold (1e7 held out) ---
arr_cf = dataset.make_split(features="raw", split="chip", fold="1e7", seed=0)
print(f"Chip-fold     : train {len(arr_cf.y_tr)}  val {len(arr_cf.y_va)}  "
      f"test {len(arr_cf.y_te)}")
print(f"  Test chips: {sorted(np.unique(arr_cf.chip_te))}")

tcount_cf = {c: int((arr_cf.chip_te == c).sum())
             for c in np.unique(arr_cf.chip_te)}
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(len(tcount_cf)), list(tcount_cf.values()), color="C3")
ax.set_xticks(range(len(tcount_cf)))
ax.set_xticklabels([c[:25] for c in tcount_cf], rotation=70, ha="right",
                   fontsize=8)
ax.set(title="Chip-fold (held-out 1e7) — only the held-out chip is in test",
       ylabel="# test pixels")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()


## 11. Model input — raw 1D vs STFT spectrogram

Six of the seven Paper 3 models take the raw 450-sample trace as a 1D
input. The seventh (the headline 2D-CNN) takes a 10×39 STFT spectrogram
instead. Below: one positive and one NTC pixel, shown both ways.

In [ ]:
i_pos = pos_idx[0]
i_neg = neg_idx[0]
x_pos, x_neg = X[i_pos], X[i_neg]
spec_pos = spec_module.transform(x_pos[None, :])[0]   # (10, 39)
spec_neg = spec_module.transform(x_neg[None, :])[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# Raw 1D
axes[0, 0].plot(t_axis, x_pos, color="C2")
axes[0, 0].set(title=f"POSITIVE pixel (raw 1D, shape {x_pos.shape})",
               xlabel="Time since onset [min]", ylabel="ΔV")
axes[0, 0].grid(alpha=0.3)
axes[0, 1].plot(t_axis, x_neg, color="C3")
axes[0, 1].set(title=f"NTC pixel (raw 1D, shape {x_neg.shape})",
               xlabel="Time since onset [min]")
axes[0, 1].grid(alpha=0.3)

# STFT spectrogram - per-pixel max-normalised to match Paper 3 Fig. 3
# panel 5/6, whose colour bar runs 0.0-1.0. (DC/low-freq is ~10^4-10^5
# bigger than mid/high, so we divide each spectrogram by its own max
# instead of using log compression — that's what the paper does.)
spec_pos_n = spec_pos / max(spec_pos.max(), 1e-12)
spec_neg_n = spec_neg / max(spec_neg.max(), 1e-12)
im = axes[1, 0].imshow(spec_pos_n, aspect="auto", origin="lower",
                       cmap="viridis", vmin=0, vmax=1)
axes[1, 0].set(title=f"POSITIVE pixel STFT (norm, shape {spec_pos.shape})",
               xlabel="STFT time bin (39)", ylabel="Frequency bin (10 lowest)")
plt.colorbar(im, ax=axes[1, 0], label="|S|^2 / max")
im2 = axes[1, 1].imshow(spec_neg_n, aspect="auto", origin="lower",
                        cmap="viridis", vmin=0, vmax=1)
axes[1, 1].set(title=f"NTC pixel STFT (norm, shape {spec_neg.shape})",
               xlabel="STFT time bin (39)")
plt.colorbar(im2, ax=axes[1, 1], label="|S|^2 / max")
plt.suptitle("What the models actually see — input representations")
plt.tight_layout()


## 11b. Paper-faithful spectrogram: non-linearised input

The shipped pipeline (Section 11) feeds the **linearised** voltage
`well.well_2d` into the STFT — that's the `X` cached by `build_dataset.py`.
Paper 3 (Tripathi 2023, Fig. 3) instead computes the spectrogram on the
**raw, non-linearised** ISFET output after only:

  1. Active-pixel mask
  2. Settled-window crop
  3. Trapped-charge subtraction (`signal - signal[0]`)

That's exactly what `well.well_2d_nl_bs_active` gives us (`_nl` = non-
linearised, `_bs` = baseline-subtracted, `_active` = active pixels only).

Below: same pixel, both ways. Linearisation is a nonlinear (log-shaped)
transform, so the two spectrograms are *not* the same signal — the
relative power of low vs. high frequencies shifts.

In [ ]:
# Locate the active-pixel column in well.well_2d_nl_bs_active that
# corresponds to the cached pixel i_pos. build_dataset uses a (looser)
# firmware + QC mask; titan's well.idx_active is stricter. Both pick
# pixels from the same underlying ordering, so we just take the i_pos-th
# column of the active subset and recompute the spectrogram from raw.
nl_active = well.well_2d_nl_bs_active   # (T, N_active), non-linearised,
                                        # settled-window, baseline-subtracted

# Crop to 450-sample window starting at time_min == 0 (the same onset
# build_dataset uses), then re-anchor at the crop start so it matches
# Paper 3 Fig. 3 panel 4 exactly.
onset_idx = int(np.searchsorted(well.time_min, 0.0))
nl_win = nl_active[onset_idx : onset_idx + 450, :]
nl_win = nl_win - nl_win[0:1, :]

# Pick the same pixel index as the linearised version (column j_pos in
# the active subset). Note: i_pos indexes into the cache (which uses the
# loose mask), not into titan's strict idx_active. To stay honest, just
# pick the first column of nl_active so we're comparing the same well's
# first active pixel both ways.
j_pos = 0
x_pos_nl = nl_win[:, j_pos]
spec_pos_nl = spec_module.transform(x_pos_nl[None, :])[0]   # (10, 39)
spec_pos_nl_n = spec_pos_nl / max(spec_pos_nl.max(), 1e-12)

# For comparison, also recompute the linearised version of THE SAME well
# from titan's strict active set so the comparison is well-matched.
lin_active = well.well_2d_bs_active     # linearised counterpart
lin_win = lin_active[onset_idx : onset_idx + 450, :]
lin_win = lin_win - lin_win[0:1, :]
x_pos_lin = lin_win[:, j_pos]
spec_pos_lin = spec_module.transform(x_pos_lin[None, :])[0]
spec_pos_lin_n = spec_pos_lin / max(spec_pos_lin.max(), 1e-12)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes[0, 0].plot(t_axis, x_pos_lin, color="C0")
axes[0, 0].set(title="Linearised (well_2d_bs_active) — as shipped to model",
               xlabel="Time since onset [min]", ylabel="dV (linearised)")
axes[0, 0].grid(alpha=0.3)
axes[0, 1].plot(t_axis, x_pos_nl, color="C1")
axes[0, 1].set(title="Non-linearised (well_2d_nl_bs_active) — paper-faithful",
               xlabel="Time since onset [min]", ylabel="dVo [V]")
axes[0, 1].grid(alpha=0.3)

im0 = axes[1, 0].imshow(spec_pos_lin_n, aspect="auto", origin="lower",
                        cmap="viridis", vmin=0, vmax=1)
axes[1, 0].set(title="STFT of linearised (norm)",
               xlabel="STFT time bin", ylabel="Freq bin")
plt.colorbar(im0, ax=axes[1, 0], label="|S|^2 / max")
im1 = axes[1, 1].imshow(spec_pos_nl_n, aspect="auto", origin="lower",
                        cmap="viridis", vmin=0, vmax=1)
axes[1, 1].set(title="STFT of non-linearised (norm) — paper Fig. 3 panel 5",
               xlabel="STFT time bin")
plt.colorbar(im1, ax=axes[1, 1], label="|S|^2 / max")
plt.suptitle(f"{CHIP_KEY} well {WELL_IDX} pixel {j_pos} — "
             "linearisation changes the STFT")
plt.tight_layout()


## 12. What the 2D-CNN sees: average spectrograms

Average STFT power across all positive vs all NTC pixels in the cache.
This is the 'signal' the 2D-CNN is supposed to learn to discriminate —
positives have a low-frequency burst at a particular time bin, NTCs
don't.

In [ ]:
specs_all = spec_module.transform(X)        # (N, 10, 39)
# Per-pixel max-normalise BEFORE averaging so each pixel contributes on
# the [0, 1] scale used by Paper 3 Fig. 3 — otherwise pixels with larger
# DC offsets dominate the mean.
specs_norm = specs_all / np.maximum(
    specs_all.max(axis=(1, 2), keepdims=True), 1e-12)
mean_pos = specs_norm[y == 1].mean(axis=0)
mean_neg = specs_norm[y == 0].mean(axis=0)
diff = mean_pos - mean_neg

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(mean_pos, aspect="auto", origin="lower",
                     cmap="viridis", vmin=0, vmax=1)
axes[0].set(title="<norm STFT> across POSITIVE pixels",
            xlabel="STFT time bin", ylabel="Frequency bin")
plt.colorbar(im0, ax=axes[0], label="mean |S|^2 / max")

im1 = axes[1].imshow(mean_neg, aspect="auto", origin="lower",
                     cmap="viridis", vmin=0, vmax=1)
axes[1].set(title="<norm STFT> across NTC pixels",
            xlabel="STFT time bin")
plt.colorbar(im1, ax=axes[1], label="mean |S|^2 / max")

im2 = axes[2].imshow(diff, aspect="auto", origin="lower",
                     cmap="RdBu_r")
axes[2].set(title="Difference (pos - neg, both normalised)",
            xlabel="STFT time bin")
plt.colorbar(im2, ax=axes[2])
plt.tight_layout()


## Summary

A single positive pixel becomes:
1. raw bin → linearised V cube (290 × 204 × T) via titan
2. firmware mask + QC pixel-level mask (typical: ~1500 surviving / 9894)
3. crop to 450 samples post-onset (~25 min wall-clock)
4. trapped-charge subtraction → starts at 0
5. cache row in `dataset_per_pixel.npz` (or `dataset_all.npz`)
6. model input — either raw 1D `(450,)` or 10×39 STFT spectrogram

Splits:
- random pixel split → known leaky (every chip in train AND test)
- chip-fold CV → leave-one-chip-out, generalisation-honest

Re-run any cell with a different `CHIP_KEY` / `WELL_IDX` at the top to
inspect a different chip / well. To re-build the underlying cache, run
`python -m Analysis.classification.data.build_dataset` (scope=final by
default) or `--scope all`.